# no hallu

In [4]:
!pip install bert-score


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
# -*- coding: utf-8 -*-

"""
analyze_no_hallu.py

Mục tiêu:
1. Đọc các file detector *_no_hallu.json.
2. Join với output generation gốc bằng:
       (chunk_id, question)
   vì một chunk_id có thể có nhiều câu hỏi.
3. Bổ sung:
   - target_hallucination_label
   - context
   - gold_answer
   - generation metadata
4. Lọc:
   - true no-hallu: is_true_no_hallu == True
   - false no-hallu: is_true_no_hallu == False
5. Thống kê:
   - theo generator
   - theo target_hallucination_label
   - tổng thể
6. QUAN TRỌNG:
   - % true no_hallu được tính trên TOÀN BỘ sample generate
   - không phải chỉ trên số sample detector predicted no_hallu
7. Xuất JSON + CSV.
"""

from pathlib import Path
import json
import re
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

# Chạy trong Jupyter Notebook
ROOT = Path.cwd()

# Nếu chạy file .py thì có thể dùng:
# ROOT = Path(__file__).resolve().parent

DATA_DIR = ROOT / "data"
DETECTOR_DIR = ROOT / "generated_hallu_detector_results"

OUTPUT_DIR = DETECTOR_DIR / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# FILES
# ============================================================

NO_HALLU_FILES = {
    "gpt5_mini":
        DETECTOR_DIR
        / "gpt5_mini_no_hallu.json",

    "deepseek_v4_flash":
        DETECTOR_DIR
        / "deepseek_v4_flash_no_hallu.json",

    "qwen_3_8_flash":
        DETECTOR_DIR
        / "qwen_3_8_flash_no_hallu.json",
}


GENERATION_FILES = {
    "gpt5_mini":
        DATA_DIR
        / "output_gpt"
        / "03_gpt5_mini_600_merged_clean.json",

    "deepseek_v4_flash":
        DATA_DIR
        / "output_deepseek2"
        / "03_deepseek_v4_flash_0731_600.json",

    "qwen_3_8_flash":
        DATA_DIR
        / "output_qwen"
        / "03_qwen_3_8_flash_600.json",
}


# ============================================================
# JSON UTILS
# ============================================================

def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file:\n{path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(
            f"File phải chứa JSON list: {path}"
        )

    return data


def save_json(data, path: Path):
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# NORMALIZE TEXT
# ============================================================

def normalize_text(text):
    """
    Chuẩn hóa text để tránh lỗi join do:
    - nhiều space
    - newline
    - tab
    - space đầu cuối
    """

    if text is None:
        return ""

    text = str(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# JOIN KEY
# ============================================================

def get_key(item):
    """
    Một chunk_id có thể có nhiều question.

    Vì vậy join bằng:
        (chunk_id, question)
    """

    chunk_id = normalize_text(
        item.get("chunk_id")
    )

    question = normalize_text(
        item.get("question")
    )

    return (
        chunk_id,
        question
    )


# ============================================================
# NORMALIZE TARGET LABEL
# ============================================================

def normalize_target_label(label):
    if label is None:
        return None

    label = normalize_text(label)

    mapping = {
        "intrinsic":
            "intrinsic_hallucination",

        "intrinsic_hallucination":
            "intrinsic_hallucination",

        "extrinsic":
            "extrinsic_hallucination",

        "extrinsic_hallucination":
            "extrinsic_hallucination",

        "mixed":
            "mixed_hallucination",

        "mixed_partial":
            "mixed_hallucination",

        "mixed_partial_hallucination":
            "mixed_hallucination",

        "mixed_hallucination":
            "mixed_hallucination",
    }

    return mapping.get(
        label,
        label
    )


# ============================================================
# GET TARGET LABEL FROM SOURCE
# ============================================================

def get_target_label_from_source(
    source_item,
    generator
):
    if not source_item:
        return None

    # --------------------------------------------------------
    # 1. Field trực tiếp
    # --------------------------------------------------------

    label = source_item.get(
        "target_hallucination_label"
    )

    if label:
        return normalize_target_label(
            label
        )

    # --------------------------------------------------------
    # 2. GPT nested structure
    # --------------------------------------------------------

    if generator == "gpt5_mini":
        gpt_generation = source_item.get(
            "gpt_generation",
            {}
        )

        if isinstance(
            gpt_generation,
            dict
        ):
            result = gpt_generation.get(
                "result",
                {}
            )

            if isinstance(
                result,
                dict
            ):
                label = result.get(
                    "target_label"
                )

                if label:
                    return normalize_target_label(
                        label
                    )

    # --------------------------------------------------------
    # 3. Generic result.target_label
    # --------------------------------------------------------

    result = source_item.get(
        "result"
    )

    if isinstance(
        result,
        dict
    ):
        label = result.get(
            "target_label"
        )

        if label:
            return normalize_target_label(
                label
            )

    return None


# ============================================================
# EXTRACT GENERATION INFO
# ============================================================

def extract_generation_info(
    source_item,
    generator
):
    if not source_item:
        return None

    # ========================================================
    # GPT
    # ========================================================

    if generator == "gpt5_mini":
        gpt_generation = source_item.get(
            "gpt_generation"
        )

        if not isinstance(
            gpt_generation,
            dict
        ):
            return None

        result = gpt_generation.get(
            "result"
        )

        if not isinstance(
            result,
            dict
        ):
            return None

        return {
            "status":
                result.get("status"),

            "answerability":
                result.get("answerability"),

            "atomic_claims_gold":
                result.get(
                    "atomic_claims_gold"
                ),

            "target_label":
                normalize_target_label(
                    result.get(
                        "target_label"
                    )
                ),

            "perturbation_plan":
                result.get(
                    "perturbation_plan"
                ),

            "operation":
                result.get(
                    "operation"
                ),

            "source_span":
                result.get(
                    "source_span"
                ),

            "hallucinated_span":
                result.get(
                    "hallucinated_span"
                ),

            "hallucinated_response":
                result.get(
                    "hallucinated_response"
                ),
        }

    # ========================================================
    # DEEPSEEK
    # ========================================================

    if generator == "deepseek_v4_flash":
        candidates = [
            source_item.get(
                "deepseek_generation"
            ),

            source_item.get(
                "generation"
            ),

            source_item.get(
                "result"
            ),
        ]

        for generation in candidates:
            if not isinstance(
                generation,
                dict
            ):
                continue

            nested_result = generation.get(
                "result"
            )

            if isinstance(
                nested_result,
                dict
            ):
                return nested_result.copy()

            return generation.copy()

        return None

    # ========================================================
    # QWEN
    # ========================================================

    if generator == "qwen_3_8_flash":
        candidates = [
            source_item.get(
                "qwen_generation"
            ),

            source_item.get(
                "generation"
            ),

            source_item.get(
                "result"
            ),
        ]

        for generation in candidates:
            if not isinstance(
                generation,
                dict
            ):
                continue

            nested_result = generation.get(
                "result"
            )

            if isinstance(
                nested_result,
                dict
            ):
                return nested_result.copy()

            return generation.copy()

        return None

    return None


# ============================================================
# BUILD GENERATION INDEX
# ============================================================

def build_generation_index(
    generator,
    data
):
    """
    Build index từ data generation đã load.

    key:
        (chunk_id, question)
    """

    index = {}

    duplicate_keys = []
    invalid_keys = []

    for item in data:
        key = get_key(item)

        chunk_id, question = key

        if not chunk_id or not question:
            invalid_keys.append(
                item
            )
            continue

        if key in index:
            duplicate_keys.append(
                key
            )

        index[key] = item

    print(
        f"[{generator}] generation source: "
        f"{len(data):,} samples"
    )

    print(
        f"[{generator}] valid index: "
        f"{len(index):,}"
    )

    if invalid_keys:
        print(
            f"[WARNING] {generator}: "
            f"{len(invalid_keys):,} sample "
            f"thiếu chunk_id hoặc question"
        )

    if duplicate_keys:
        print(
            f"[WARNING] {generator}: "
            f"{len(duplicate_keys):,} duplicate "
            f"(chunk_id, question)"
        )

        print(
            "Ví dụ duplicate:"
        )

        for key in duplicate_keys[:5]:
            print(
                "  chunk_id =",
                key[0]
            )

            print(
                "  question =",
                key[1]
            )

            print()

    return index


# ============================================================
# GET GENERATED ANSWER FROM SOURCE
# ============================================================

def get_generated_answer_from_source(
    source_item,
    generator
):
    if not source_item:
        return None

    # --------------------------------------------------------
    # GPT
    # --------------------------------------------------------

    if generator == "gpt5_mini":
        result = (
            source_item
            .get(
                "gpt_generation",
                {}
            )
            .get(
                "result",
                {}
            )
        )

        if isinstance(
            result,
            dict
        ):
            return (
                result.get(
                    "hallucinated_response"
                )
                or result.get(
                    "answer"
                )
            )

    # --------------------------------------------------------
    # Generic fields
    # --------------------------------------------------------

    possible_fields = [
        "hallucinated_answer",
        "hallucinated_response",
        "generated_answer",
        "answer_generated",
        "response",
    ]

    for field in possible_fields:
        value = source_item.get(
            field
        )

        if value:
            return value

    # --------------------------------------------------------
    # Nested generation
    # --------------------------------------------------------

    for key in [
        "deepseek_generation",
        "qwen_generation",
        "generation",
        "result",
    ]:
        obj = source_item.get(
            key
        )

        if not isinstance(
            obj,
            dict
        ):
            continue

        nested = obj.get(
            "result"
        )

        if isinstance(
            nested,
            dict
        ):
            obj = nested

        for field in possible_fields:
            value = obj.get(
                field
            )

            if value:
                return value

    return None


# ============================================================
# ENRICH ONE SAMPLE
# ============================================================

def enrich_sample(
    sample,
    generator,
    source_item
):
    row = sample.copy()

    # --------------------------------------------------------
    # generator
    # --------------------------------------------------------

    row["generator"] = (
        row.get("generator")
        or generator
    )

    # --------------------------------------------------------
    # context
    # --------------------------------------------------------

    if source_item:
        if not row.get("context"):
            row["context"] = (
                source_item.get(
                    "context"
                )
                or source_item.get(
                    "content"
                )
            )

    # --------------------------------------------------------
    # question
    # --------------------------------------------------------

    if source_item:
        if not row.get("question"):
            row["question"] = (
                source_item.get(
                    "question"
                )
            )

    # --------------------------------------------------------
    # gold answer
    # --------------------------------------------------------

    if source_item:
        if not row.get("gold_answer"):
            row["gold_answer"] = (
                source_item.get(
                    "gold_answer"
                )
                or source_item.get(
                    "answer"
                )
            )

    # --------------------------------------------------------
    # target label
    # --------------------------------------------------------

    target = row.get(
        "target_hallucination_label"
    )

    if not target:
        target = get_target_label_from_source(
            source_item,
            generator
        )

    row[
        "target_hallucination_label"
    ] = normalize_target_label(
        target
    )

    # --------------------------------------------------------
    # hallucinated answer
    # --------------------------------------------------------

    if not row.get(
        "hallucinated_answer"
    ):
        generated_answer = (
            get_generated_answer_from_source(
                source_item,
                generator
            )
        )

        if generated_answer:
            row[
                "hallucinated_answer"
            ] = generated_answer

    # --------------------------------------------------------
    # generation info
    # --------------------------------------------------------

    generation_info = (
        extract_generation_info(
            source_item,
            generator
        )
    )

    if generation_info:
        row[
            "generation_info"
        ] = generation_info

    return row


# ============================================================
# PROCESS ONE GENERATOR
# ============================================================

def process_generator(
    generator
):
    print()
    print("=" * 100)

    print(
        f"PROCESSING: {generator}"
    )

    print("=" * 100)

    detector_path = (
        NO_HALLU_FILES[
            generator
        ]
    )

    generation_path = (
        GENERATION_FILES[
            generator
        ]
    )

    # ========================================================
    # LOAD DATA
    # ========================================================

    detector_data = load_json(
        detector_path
    )

    generation_data = load_json(
        generation_path
    )

    # Tổng sample đã generate
    total_generated = len(
        generation_data
    )

    generation_index = (
        build_generation_index(
            generator,
            generation_data
        )
    )

    enriched = []

    missing_generation = []
    missing_target = []

    # ========================================================
    # JOIN
    # ========================================================

    for sample in detector_data:
        key = get_key(
            sample
        )

        source_item = (
            generation_index.get(
                key
            )
        )

        if source_item is None:
            missing_generation.append(
                {
                    "chunk_id":
                        sample.get(
                            "chunk_id"
                        ),

                    "question":
                        sample.get(
                            "question"
                        ),
                }
            )

        row = enrich_sample(
            sample,
            generator,
            source_item
        )

        if not row.get(
            "target_hallucination_label"
        ):
            missing_target.append(
                {
                    "chunk_id":
                        row.get(
                            "chunk_id"
                        ),

                    "question":
                        row.get(
                            "question"
                        ),
                }
            )

        enriched.append(
            row
        )

    # ========================================================
    # SPLIT TRUE / FALSE
    # ========================================================

    # TRUE:
    # detector predicted no_hallu
    # review xác nhận thực sự no_hallu
    # => generator FAILED tạo hallucination

    true_no_hallu = [
        x
        for x in enriched
        if x.get(
            "is_true_no_hallu"
        ) is True
    ]

    # FALSE:
    # detector predicted no_hallu
    # nhưng review xác nhận vẫn có hallucination
    # => detector sai, generator vẫn tạo hallu

    false_no_hallu = [
        x
        for x in enriched
        if x.get(
            "is_true_no_hallu"
        ) is False
    ]

    # Chưa review / field không có True False
    unknown_review = [
        x
        for x in enriched
        if x.get(
            "is_true_no_hallu"
        ) not in [
            True,
            False
        ]
    ]

    # ========================================================
    # SAVE INDIVIDUAL FILES
    # ========================================================

    save_json(
        enriched,
        OUTPUT_DIR
        / f"{generator}_no_hallu_enriched.json"
    )

    save_json(
        true_no_hallu,
        OUTPUT_DIR
        / f"{generator}_true_no_hallu.json"
    )

    save_json(
        false_no_hallu,
        OUTPUT_DIR
        / f"{generator}_false_no_hallu.json"
    )

    save_json(
        unknown_review,
        OUTPUT_DIR
        / f"{generator}_unknown_review.json"
    )

    save_json(
        missing_generation,
        OUTPUT_DIR
        / f"{generator}_missing_generation_join.json"
    )

    save_json(
        missing_target,
        OUTPUT_DIR
        / f"{generator}_missing_target_label.json"
    )

    # ========================================================
    # STATS
    # ========================================================

    # Số detector predicted no_hallu
    n_detector_no_hallu = len(
        enriched
    )

    # Trong số đó, review xác nhận
    n_true = len(
        true_no_hallu
    )

    n_false = len(
        false_no_hallu
    )

    n_unknown = len(
        unknown_review
    )

    # ========================================================
    # % TRÊN TOÀN BỘ SAMPLE GENERATED
    # ========================================================

    detector_no_hallu_pct_of_all = (
        n_detector_no_hallu
        / total_generated
        * 100
        if total_generated
        else 0
    )

    # QUAN TRỌNG NHẤT:
    # generator thật sự sinh no_hallu
    # / toàn bộ sample generate

    true_no_hallu_pct_of_all = (
        n_true
        / total_generated
        * 100
        if total_generated
        else 0
    )

    # Detector đoán no_hallu nhưng thực tế có hallu
    # / toàn bộ sample generate

    false_no_hallu_pct_of_all = (
        n_false
        / total_generated
        * 100
        if total_generated
        else 0
    )

    unknown_pct_of_all = (
        n_unknown
        / total_generated
        * 100
        if total_generated
        else 0
    )

    # ========================================================
    # % CHỈ TRONG TẬP DETECTOR FLAG NO_HALLU
    # chỉ để tham khảo
    # ========================================================

    true_pct_among_flagged = (
        n_true
        / n_detector_no_hallu
        * 100
        if n_detector_no_hallu
        else 0
    )

    false_pct_among_flagged = (
        n_false
        / n_detector_no_hallu
        * 100
        if n_detector_no_hallu
        else 0
    )

    unknown_pct_among_flagged = (
        n_unknown
        / n_detector_no_hallu
        * 100
        if n_detector_no_hallu
        else 0
    )

    # ========================================================
    # PRINT
    # ========================================================

    print()

    print(
        f"Total generated             : "
        f"{total_generated:,}"
    )

    print(
        f"Detector predicted no_hallu : "
        f"{n_detector_no_hallu:,} "
        f"({detector_no_hallu_pct_of_all:.2f}% of ALL)"
    )

    print(
        f"TRUE no_hallu               : "
        f"{n_true:,} "
        f"({true_no_hallu_pct_of_all:.2f}% of ALL)"
    )

    print(
        f"False detector no_hallu     : "
        f"{n_false:,} "
        f"({false_no_hallu_pct_of_all:.2f}% of ALL)"
    )

    print(
        f"Unknown review              : "
        f"{n_unknown:,} "
        f"({unknown_pct_of_all:.2f}% of ALL)"
    )

    print()

    print(
        "--- Among detector no_hallu only ---"
    )

    print(
        f"Actually no_hallu           : "
        f"{true_pct_among_flagged:.2f}%"
    )

    print(
        f"Actually hallucination      : "
        f"{false_pct_among_flagged:.2f}%"
    )

    print(
        f"Unknown                     : "
        f"{unknown_pct_among_flagged:.2f}%"
    )

    print()

    print(
        f"Missing generation join     : "
        f"{len(missing_generation):,}"
    )

    print(
        f"Missing target label        : "
        f"{len(missing_target):,}"
    )

    # ========================================================
    # RETURN
    # ========================================================

    return {
        "generator":
            generator,

        "samples":
            enriched,

        "true_no_hallu":
            true_no_hallu,

        "false_no_hallu":
            false_no_hallu,

        "unknown_review":
            unknown_review,

        # Tổng sample thật sự generate
        "total_generated":
            total_generated,

        # Detector predicted no_hallu
        "n_detector_no_hallu":
            n_detector_no_hallu,

        "detector_no_hallu_pct_of_all":
            detector_no_hallu_pct_of_all,

        # TRUE no_hallu
        "n_true_no_hallu":
            n_true,

        "true_no_hallu_pct_of_all":
            true_no_hallu_pct_of_all,

        # FALSE no_hallu
        "n_false_no_hallu":
            n_false,

        "false_no_hallu_pct_of_all":
            false_no_hallu_pct_of_all,

        # Unknown
        "n_unknown_review":
            n_unknown,

        "unknown_pct_of_all":
            unknown_pct_of_all,

        # Trong detector flagged set
        "true_pct_among_flagged":
            true_pct_among_flagged,

        "false_pct_among_flagged":
            false_pct_among_flagged,

        "unknown_pct_among_flagged":
            unknown_pct_among_flagged,

        "missing_generation":
            len(
                missing_generation
            ),

        "missing_target_label":
            len(
                missing_target
            ),
    }


# ============================================================
# TARGET LABEL STATS
# ============================================================

def build_target_stats(
    all_samples,
    generation_data_by_model
):
    """
    Thống kê target label theo từng generator.

    Quan trọng:
    mẫu số của % failure theo target
    = toàn bộ sample generation của target đó,
    KHÔNG phải chỉ detector no_hallu.
    """

    rows = []

    for generator, generation_data in generation_data_by_model.items():

        detector_samples = [
            x
            for x in all_samples
            if x.get(
                "generator"
            ) == generator
        ]

        # ----------------------------------------------------
        # Đếm tổng target label trong ALL generation data
        # ----------------------------------------------------

        total_by_target = {}

        for item in generation_data:
            label = get_target_label_from_source(
                item,
                generator
            )

            label = (
                label
                or "missing"
            )

            total_by_target[
                label
            ] = (
                total_by_target.get(
                    label,
                    0
                )
                + 1
            )

        labels = sorted(
            total_by_target.keys()
        )

        # ----------------------------------------------------
        # Với mỗi target
        # ----------------------------------------------------

        for label in labels:

            total_generated_target = (
                total_by_target[
                    label
                ]
            )

            flagged_subset = [
                x
                for x in detector_samples
                if (
                    x.get(
                        "target_hallucination_label"
                    )
                    or "missing"
                ) == label
            ]

            n_detector_flagged = len(
                flagged_subset
            )

            n_true = sum(
                x.get(
                    "is_true_no_hallu"
                ) is True
                for x in flagged_subset
            )

            n_false = sum(
                x.get(
                    "is_true_no_hallu"
                ) is False
                for x in flagged_subset
            )

            n_unknown = (
                n_detector_flagged
                - n_true
                - n_false
            )

            # -----------------------------------------------
            # % ALL của target
            # -----------------------------------------------

            detector_pct_of_target = (
                n_detector_flagged
                / total_generated_target
                * 100
                if total_generated_target
                else 0
            )

            true_no_hallu_pct_of_target = (
                n_true
                / total_generated_target
                * 100
                if total_generated_target
                else 0
            )

            false_no_hallu_pct_of_target = (
                n_false
                / total_generated_target
                * 100
                if total_generated_target
                else 0
            )

            rows.append(
                {
                    "generator":
                        generator,

                    "target_hallucination_label":
                        label,

                    "total_generated_target":
                        total_generated_target,

                    "detector_no_hallu_count":
                        n_detector_flagged,

                    "detector_no_hallu_pct_of_target":
                        detector_pct_of_target,

                    "true_no_hallu_count":
                        n_true,

                    # QUAN TRỌNG
                    "true_no_hallu_pct_of_target":
                        true_no_hallu_pct_of_target,

                    "false_detector_no_hallu_count":
                        n_false,

                    "false_detector_no_hallu_pct_of_target":
                        false_no_hallu_pct_of_target,

                    "unknown_review_count":
                        n_unknown,
                }
            )

    return pd.DataFrame(
        rows
    )


# ============================================================
# TARGET LABEL OVERALL
# ============================================================

def build_target_overall_stats(
    all_samples,
    generation_data_by_model
):
    """
    Gộp tất cả generator rồi tính theo target label.
    """

    total_by_target = {}

    # ========================================================
    # Tổng sample generate theo target
    # ========================================================

    for generator, generation_data in generation_data_by_model.items():

        for item in generation_data:

            label = get_target_label_from_source(
                item,
                generator
            )

            label = (
                label
                or "missing"
            )

            total_by_target[
                label
            ] = (
                total_by_target.get(
                    label,
                    0
                )
                + 1
            )

    rows = []

    for label in sorted(
        total_by_target.keys()
    ):

        total_generated_target = (
            total_by_target[
                label
            ]
        )

        flagged_subset = [
            x
            for x in all_samples
            if (
                x.get(
                    "target_hallucination_label"
                )
                or "missing"
            ) == label
        ]

        n_detector_flagged = len(
            flagged_subset
        )

        n_true = sum(
            x.get(
                "is_true_no_hallu"
            ) is True
            for x in flagged_subset
        )

        n_false = sum(
            x.get(
                "is_true_no_hallu"
            ) is False
            for x in flagged_subset
        )

        n_unknown = (
            n_detector_flagged
            - n_true
            - n_false
        )

        rows.append(
            {
                "target_hallucination_label":
                    label,

                "total_generated_target":
                    total_generated_target,

                "detector_no_hallu_count":
                    n_detector_flagged,

                "detector_no_hallu_pct_of_target":
                    (
                        n_detector_flagged
                        / total_generated_target
                        * 100
                        if total_generated_target
                        else 0
                    ),

                "true_no_hallu_count":
                    n_true,

                "true_no_hallu_pct_of_target":
                    (
                        n_true
                        / total_generated_target
                        * 100
                        if total_generated_target
                        else 0
                    ),

                "false_detector_no_hallu_count":
                    n_false,

                "false_detector_no_hallu_pct_of_target":
                    (
                        n_false
                        / total_generated_target
                        * 100
                        if total_generated_target
                        else 0
                    ),

                "unknown_review_count":
                    n_unknown,
            }
        )

    return pd.DataFrame(
        rows
    )


# ============================================================
# MAIN
# ============================================================

def main():

    results = []

    all_samples = []
    all_true_no_hallu = []
    all_false_no_hallu = []
    all_unknown_review = []

    generation_data_by_model = {}

    # ========================================================
    # LOAD ALL GENERATION FILES
    # ========================================================

    for generator, path in GENERATION_FILES.items():
        generation_data_by_model[
            generator
        ] = load_json(
            path
        )

    # ========================================================
    # PROCESS ALL GENERATORS
    # ========================================================

    for generator in NO_HALLU_FILES:

        result = process_generator(
            generator
        )

        results.append(
            result
        )

        all_samples.extend(
            result[
                "samples"
            ]
        )

        all_true_no_hallu.extend(
            result[
                "true_no_hallu"
            ]
        )

        all_false_no_hallu.extend(
            result[
                "false_no_hallu"
            ]
        )

        all_unknown_review.extend(
            result[
                "unknown_review"
            ]
        )

    # ========================================================
    # SAVE COMBINED JSON
    # ========================================================

    save_json(
        all_samples,
        OUTPUT_DIR
        / "all_no_hallu_enriched.json"
    )

    save_json(
        all_true_no_hallu,
        OUTPUT_DIR
        / "all_true_no_hallu.json"
    )

    save_json(
        all_false_no_hallu,
        OUTPUT_DIR
        / "all_false_no_hallu.json"
    )

    save_json(
        all_unknown_review,
        OUTPUT_DIR
        / "all_unknown_review.json"
    )

    # ========================================================
    # GENERATOR SUMMARY
    # ========================================================

    summary_rows = []

    for r in results:

        summary_rows.append(
            {
                "generator":
                    r[
                        "generator"
                    ],

                "total_generated":
                    r[
                        "total_generated"
                    ],

                "detector_no_hallu_count":
                    r[
                        "n_detector_no_hallu"
                    ],

                "detector_no_hallu_pct_of_all":
                    r[
                        "detector_no_hallu_pct_of_all"
                    ],

                "true_no_hallu_count":
                    r[
                        "n_true_no_hallu"
                    ],

                # QUAN TRỌNG NHẤT
                "true_no_hallu_pct_of_all":
                    r[
                        "true_no_hallu_pct_of_all"
                    ],

                "false_detector_no_hallu_count":
                    r[
                        "n_false_no_hallu"
                    ],

                "false_detector_no_hallu_pct_of_all":
                    r[
                        "false_no_hallu_pct_of_all"
                    ],

                "unknown_review_count":
                    r[
                        "n_unknown_review"
                    ],

                "unknown_pct_of_all":
                    r[
                        "unknown_pct_of_all"
                    ],

                "true_pct_among_detector_flagged":
                    r[
                        "true_pct_among_flagged"
                    ],

                "false_pct_among_detector_flagged":
                    r[
                        "false_pct_among_flagged"
                    ],

                "missing_generation_join":
                    r[
                        "missing_generation"
                    ],

                "missing_target_label":
                    r[
                        "missing_target_label"
                    ],
            }
        )

    generator_summary = pd.DataFrame(
        summary_rows
    )

    generator_summary.to_csv(
        OUTPUT_DIR
        / "generator_no_hallu_summary.csv",

        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # TARGET LABEL BY GENERATOR
    # ========================================================

    target_by_generator = (
        build_target_stats(
            all_samples,
            generation_data_by_model
        )
    )

    target_by_generator.to_csv(
        OUTPUT_DIR
        / "target_label_by_generator.csv",

        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # TARGET LABEL OVERALL
    # ========================================================

    target_overall = (
        build_target_overall_stats(
            all_samples,
            generation_data_by_model
        )
    )

    target_overall.to_csv(
        OUTPUT_DIR
        / "target_label_overall.csv",

        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # OVERALL
    # ========================================================

    total_generated_all = sum(
        r[
            "total_generated"
        ]
        for r in results
    )

    total_detector_no_hallu = len(
        all_samples
    )

    n_true = len(
        all_true_no_hallu
    )

    n_false = len(
        all_false_no_hallu
    )

    n_unknown = len(
        all_unknown_review
    )

    overall = {
        "total_generated":
            total_generated_all,

        "detector_no_hallu_count":
            total_detector_no_hallu,

        "detector_no_hallu_pct_of_all":
            (
                total_detector_no_hallu
                / total_generated_all
                * 100
                if total_generated_all
                else 0
            ),

        "true_no_hallu_count":
            n_true,

        # QUAN TRỌNG NHẤT:
        # true no_hallu / ALL generated
        "true_no_hallu_pct_of_all":
            (
                n_true
                / total_generated_all
                * 100
                if total_generated_all
                else 0
            ),

        "false_detector_no_hallu_count":
            n_false,

        "false_detector_no_hallu_pct_of_all":
            (
                n_false
                / total_generated_all
                * 100
                if total_generated_all
                else 0
            ),

        "unknown_review_count":
            n_unknown,

        "unknown_pct_of_all":
            (
                n_unknown
                / total_generated_all
                * 100
                if total_generated_all
                else 0
            ),
    }

    save_json(
        overall,
        OUTPUT_DIR
        / "overall_summary.json"
    )

    # ========================================================
    # PRINT GENERATOR SUMMARY
    # ========================================================

    print()
    print()
    print("=" * 120)

    print(
        "GENERATOR SUMMARY"
    )

    print("=" * 120)

    print(
        generator_summary.to_string(
            index=False
        )
    )

    # ========================================================
    # PRINT TARGET BY GENERATOR
    # ========================================================

    print()
    print()
    print("=" * 120)

    print(
        "TARGET LABEL BY GENERATOR"
    )

    print("=" * 120)

    print(
        target_by_generator.to_string(
            index=False
        )
    )

    # ========================================================
    # PRINT TARGET OVERALL
    # ========================================================

    print()
    print()
    print("=" * 120)

    print(
        "TARGET LABEL OVERALL"
    )

    print("=" * 120)

    print(
        target_overall.to_string(
            index=False
        )
    )

    # ========================================================
    # PRINT OVERALL
    # ========================================================

    print()
    print()
    print("=" * 120)

    print(
        "OVERALL - ALL GENERATED SAMPLES"
    )

    print("=" * 120)

    print(
        f"Total generated             : "
        f"{total_generated_all:,}"
    )

    print(
        f"Detector predicted no_hallu : "
        f"{total_detector_no_hallu:,} "
        f"({overall['detector_no_hallu_pct_of_all']:.2f}% of ALL)"
    )

    print(
        f"TRUE no_hallu               : "
        f"{n_true:,} "
        f"({overall['true_no_hallu_pct_of_all']:.2f}% of ALL)"
    )

    print(
        f"False detector no_hallu     : "
        f"{n_false:,} "
        f"({overall['false_detector_no_hallu_pct_of_all']:.2f}% of ALL)"
    )

    print(
        f"Unknown                     : "
        f"{n_unknown:,} "
        f"({overall['unknown_pct_of_all']:.2f}% of ALL)"
    )

    print()

    print(
        "Output directory:"
    )

    print(
        OUTPUT_DIR
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()


PROCESSING: gpt5_mini
[gpt5_mini] generation source: 580 samples
[gpt5_mini] valid index: 580

Total generated             : 580
Detector predicted no_hallu : 75 (12.93% of ALL)
TRUE no_hallu               : 1 (0.17% of ALL)
False detector no_hallu     : 74 (12.76% of ALL)
Unknown review              : 0 (0.00% of ALL)

--- Among detector no_hallu only ---
Actually no_hallu           : 1.33%
Actually hallucination      : 98.67%
Unknown                     : 0.00%

Missing generation join     : 0
Missing target label        : 0

PROCESSING: deepseek_v4_flash
[deepseek_v4_flash] generation source: 600 samples
[deepseek_v4_flash] valid index: 600

Total generated             : 600
Detector predicted no_hallu : 117 (19.50% of ALL)
TRUE no_hallu               : 15 (2.50% of ALL)
False detector no_hallu     : 102 (17.00% of ALL)
Unknown review              : 0 (0.00% of ALL)

--- Among detector no_hallu only ---
Actually no_hallu           : 12.82%
Actually hallucination      : 87.18%
Unkno

# diem

In [10]:
# -*- coding: utf-8 -*-

"""
select_generator_no_judge.py

Chọn LLM tốt nhất để sinh guided hallucination
KHÔNG sử dụng LLM Judge.

Metrics:
1. Generation failure rate:
       true_no_hallu / all generated
   -> thấp tốt

2. ROUGE-L Gold vs Hallucinated
   -> cao tốt

3. BERTScore F1 Gold vs Hallucinated
   -> cao tốt

4. Minimal perturbation:
   - lexical similarity
   - token change ratio
   -> lexical similarity cao / change ratio thấp tốt

5. Detector miss proxy:
       confirmed hallucination nhưng detector predicted no_hallu
       / estimated hallucination samples
   -> cao tốt

6. Detector miss theo hallucination type:
       detector miss(type)
       /
       [total generated(type) - true_no_hallu(type)]

   Ví dụ:
       intrinsic:
       - generated = 200
       - true_no_hallu = 5
       - detector miss = 30

       true hallucination ~= 195
       miss rate = 30 / 195

7. Diversity:
   heuristic lexical change.
   Chỉ dùng tham khảo.

Cuối cùng:
- normalize metric giữa các generator
- tính final_score
- rank generator
- chọn generator tốt nhất
"""

from pathlib import Path
import json
import re
import math
from collections import Counter
from difflib import SequenceMatcher

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

# Notebook
ROOT = Path.cwd()

# Nếu chạy file .py và file nằm ở root project:
# ROOT = Path(__file__).resolve().parent


DATA_DIR = (
    ROOT
    / "data"
)


DETECTOR_DIR = (
    ROOT
    / "generated_hallu_detector_results"
)


ANALYSIS_DIR = (
    DETECTOR_DIR
    / "analysis"
)


OUTPUT_DIR = (
    ANALYSIS_DIR
    / "generator_selection"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# GENERATION FILES
# ============================================================

GENERATION_FILES = {

    "gpt5_mini":
        DATA_DIR
        / "output_gpt"
        / "03_gpt5_mini_600_merged_clean.json",

    "deepseek_v4_flash":
        DATA_DIR
        / "output_deepseek2"
        / "03_deepseek_v4_flash_0731_600.json",

    "qwen_3_8_flash":
        DATA_DIR
        / "output_qwen"
        / "03_qwen_3_8_flash_600.json",
}


# ============================================================
# TRUE / FALSE NO-HALLU
# ============================================================

TRUE_NO_HALLU_FILES = {

    "gpt5_mini":
        ANALYSIS_DIR
        / "gpt5_mini_true_no_hallu.json",

    "deepseek_v4_flash":
        ANALYSIS_DIR
        / "deepseek_v4_flash_true_no_hallu.json",

    "qwen_3_8_flash":
        ANALYSIS_DIR
        / "qwen_3_8_flash_true_no_hallu.json",
}


FALSE_NO_HALLU_FILES = {

    "gpt5_mini":
        ANALYSIS_DIR
        / "gpt5_mini_false_no_hallu.json",

    "deepseek_v4_flash":
        ANALYSIS_DIR
        / "deepseek_v4_flash_false_no_hallu.json",

    "qwen_3_8_flash":
        ANALYSIS_DIR
        / "qwen_3_8_flash_false_no_hallu.json",
}


# ============================================================
# BERTSCORE
# ============================================================

COMPUTE_BERTSCORE = True

BERTSCORE_BATCH_SIZE = 16


# ============================================================
# FINAL SCORE WEIGHTS
#
# Giữ weighting hiện tại để không thay đổi kết quả experiment.
# Diversity vẫn để lại như script cũ.
# ============================================================

WEIGHTS = {

    "generation_success":
        0.30,

    "semantic_similarity":
        0.25,

    "detector_difficulty":
        0.20,

    "minimality":
        0.15,

    "diversity":
        0.10,
}


# ============================================================
# JSON UTILS
# ============================================================

def load_json(path):

    if not path.exists():

        print(
            f"[WARNING] Không tìm thấy: {path}"
        )

        return []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    if not isinstance(
        data,
        list
    ):

        raise ValueError(
            f"Expected JSON list: {path}"
        )

    return data


def save_json(
    data,
    path
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# NORMALIZE TEXT
# ============================================================

def normalize_text(text):

    if text is None:
        return ""

    text = str(
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def tokenize(text):

    text = normalize_text(
        text
    ).lower()

    return re.findall(
        r"\w+|[^\w\s]",
        text,
        flags=re.UNICODE
    )


# ============================================================
# RECURSIVE FIND
# ============================================================

def recursive_find_first(
    obj,
    candidate_keys
):

    if isinstance(
        obj,
        dict
    ):

        # ưu tiên current level
        for key in candidate_keys:

            value = obj.get(
                key
            )

            if (
                value is not None
                and value != ""
            ):

                return value

        # recursive
        for value in obj.values():

            found = recursive_find_first(
                value,
                candidate_keys
            )

            if found is not None:
                return found

    elif isinstance(
        obj,
        list
    ):

        for value in obj:

            found = recursive_find_first(
                value,
                candidate_keys
            )

            if found is not None:
                return found

    return None


# ============================================================
# GET GOLD ANSWER
# ============================================================

def get_gold_answer(item):

    for key in [
        "gold_answer",
        "answer",
    ]:

        value = item.get(
            key
        )

        if value:

            return normalize_text(
                value
            )

    return ""


# ============================================================
# GET HALLUCINATED ANSWER
# ============================================================

def get_hallucinated_answer(item):

    candidate_keys = [

        "hallucinated_answer",

        "hallucinated_response",

        "generated_answer",

        "answer_generated",

        "generated_response",

        "model_answer",
    ]

    value = recursive_find_first(
        item,
        candidate_keys
    )

    if value:

        return normalize_text(
            value
        )

    return ""


# ============================================================
# NORMALIZE HALLUCINATION LABEL
# ============================================================

def normalize_target_label(label):

    if label is None:
        return "unknown"

    label = normalize_text(
        label
    )

    mapping = {

        "intrinsic":
            "intrinsic_hallucination",

        "intrinsic_hallucination":
            "intrinsic_hallucination",

        "extrinsic":
            "extrinsic_hallucination",

        "extrinsic_hallucination":
            "extrinsic_hallucination",

        "mixed":
            "mixed_hallucination",

        "mixed_partial":
            "mixed_hallucination",

        "mixed_partial_hallucination":
            "mixed_hallucination",

        "mixed_hallucination":
            "mixed_hallucination",
    }

    return mapping.get(
        label,
        label
    )


# ============================================================
# GET TARGET LABEL
# ============================================================

def get_target_label(item):

    value = recursive_find_first(
        item,
        [
            "target_hallucination_label",
            "target_label",
        ]
    )

    if value is None:
        return "unknown"

    return normalize_target_label(
        value
    )


# ============================================================
# ROUGE-L
# ============================================================

def lcs_length(
    x,
    y
):

    if not x or not y:
        return 0

    previous = [
        0
    ] * (
        len(y) + 1
    )

    for a in x:

        current = [
            0
        ]

        for j, b in enumerate(
            y,
            start=1
        ):

            if a == b:

                current.append(
                    previous[
                        j - 1
                    ] + 1
                )

            else:

                current.append(
                    max(
                        previous[j],
                        current[
                            j - 1
                        ]
                    )
                )

        previous = current

    return previous[-1]


def rouge_l_f1(
    gold,
    pred
):

    gold_tokens = tokenize(
        gold
    )

    pred_tokens = tokenize(
        pred
    )

    if (
        not gold_tokens
        or not pred_tokens
    ):

        return 0.0

    lcs = lcs_length(
        gold_tokens,
        pred_tokens
    )

    precision = (
        lcs
        / len(pred_tokens)
    )

    recall = (
        lcs
        / len(gold_tokens)
    )

    if (
        precision + recall
        == 0
    ):

        return 0.0

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )


# ============================================================
# LEXICAL SIMILARITY
# ============================================================

def lexical_similarity(
    gold,
    pred
):

    a = tokenize(
        gold
    )

    b = tokenize(
        pred
    )

    if not a or not b:
        return 0.0

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()


# ============================================================
# TOKEN CHANGE RATIO
# ============================================================

def token_change_ratio(
    gold,
    pred
):

    return (
        1.0
        - lexical_similarity(
            gold,
            pred
        )
    )


# ============================================================
# LENGTH RATIO
# ============================================================

def length_ratio(
    gold,
    pred
):

    g = len(
        tokenize(
            gold
        )
    )

    p = len(
        tokenize(
            pred
        )
    )

    if g == 0:
        return 0.0

    return (
        p
        / g
    )


# ============================================================
# CHANGE TYPE HEURISTIC
# ============================================================

NUMBER_RE = re.compile(
    r"\b\d+(?:[.,]\d+)?\b"
)


DATE_RE = re.compile(
    r"\b(?:"
    r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}"
    r"|"
    r"(?:19|20)\d{2}"
    r")\b"
)


NEGATION_WORDS = {
    "không",
    "chưa",
    "chẳng",
    "không phải",
    "không được",
    "không có",
}


QUANTIFIER_WORDS = {
    "tất cả",
    "mọi",
    "hầu hết",
    "phần lớn",
    "một số",
    "nhiều",
    "ít",
    "duy nhất",
    "chỉ",
    "toàn bộ",
    "đa số",
}


def changed_tokens(
    gold,
    pred
):

    a = tokenize(
        gold
    )

    b = tokenize(
        pred
    )

    matcher = SequenceMatcher(
        None,
        a,
        b
    )

    removed = []
    added = []
    operations = []

    for (
        tag,
        i1,
        i2,
        j1,
        j2
    ) in matcher.get_opcodes():

        if tag == "equal":
            continue

        operations.append(
            tag
        )

        if tag in [
            "delete",
            "replace"
        ]:

            removed.extend(
                a[
                    i1:i2
                ]
            )

        if tag in [
            "insert",
            "replace"
        ]:

            added.extend(
                b[
                    j1:j2
                ]
            )

    return (
        removed,
        added,
        operations
    )


def infer_change_type(
    gold,
    pred
):

    (
        removed,
        added,
        operations
    ) = changed_tokens(
        gold,
        pred
    )

    removed_text = " ".join(
        removed
    ).lower()

    added_text = " ".join(
        added
    ).lower()

    changed_text = (
        removed_text
        + " "
        + added_text
    )

    # --------------------------------------------------------
    # NUMBER / DATE
    # --------------------------------------------------------

    if NUMBER_RE.search(
        changed_text
    ):

        if DATE_RE.search(
            changed_text
        ):
            return "date"

        return "number"

    # --------------------------------------------------------
    # NEGATION
    # --------------------------------------------------------

    for word in NEGATION_WORDS:

        if (
            word in removed_text
            or word in added_text
        ):

            return "negation"

    # --------------------------------------------------------
    # QUANTIFIER
    # --------------------------------------------------------

    for word in QUANTIFIER_WORDS:

        if (
            word in removed_text
            or word in added_text
        ):

            return "quantifier"

    # --------------------------------------------------------
    # OPERATIONS
    # --------------------------------------------------------

    unique_ops = set(
        operations
    )

    if unique_ops == {
        "insert"
    }:

        return "addition"

    if unique_ops == {
        "delete"
    }:

        return "deletion"

    if (
        "replace" in unique_ops
        and len(removed) <= 5
        and len(added) <= 5
    ):

        return "entity_or_attribute"

    if len(
        unique_ops
    ) > 1:

        return "mixed_edit"

    return "lexical_other"


# ============================================================
# NORMALIZED ENTROPY
# ============================================================

def normalized_entropy(categories):

    if not categories:
        return 0.0

    counts = Counter(
        categories
    )

    total = sum(
        counts.values()
    )

    if len(
        counts
    ) <= 1:

        return 0.0

    probs = [
        c / total
        for c
        in counts.values()
    ]

    entropy = -sum(
        p * math.log(p)
        for p
        in probs
        if p > 0
    )

    max_entropy = math.log(
        len(counts)
    )

    if max_entropy == 0:
        return 0.0

    return (
        entropy
        / max_entropy
    )


# ============================================================
# BERTSCORE
# ============================================================

def compute_bertscore(
    golds,
    preds
):

    if not COMPUTE_BERTSCORE:

        return [
            np.nan
        ] * len(
            golds
        )

    try:

        from bert_score import score

    except ImportError:

        print()
        print(
            "[WARNING] bert_score chưa được cài."
        )

        print(
            "Cài bằng:"
        )

        print(
            "pip install bert-score"
        )

        print(
            "Bỏ qua BERTScore."
        )

        return [
            np.nan
        ] * len(
            golds
        )

    print()
    print(
        "Computing BERTScore..."
    )

    _, _, f1 = score(
        preds,
        golds,
        lang="vi",
        batch_size=BERTSCORE_BATCH_SIZE,
        verbose=True
    )

    return (
        f1
        .cpu()
        .numpy()
        .tolist()
    )


# ============================================================
# PROCESS GENERATION DATA
# ============================================================

def process_generation_data(
    generator,
    data
):

    rows = []

    skipped = 0

    for item in data:

        gold = get_gold_answer(
            item
        )

        hallu = get_hallucinated_answer(
            item
        )

        if (
            not gold
            or not hallu
        ):

            skipped += 1
            continue

        row = {

            "generator":
                generator,

            "chunk_id":
                item.get(
                    "chunk_id"
                ),

            "question":
                item.get(
                    "question"
                ),

            "target_hallucination_label":
                get_target_label(
                    item
                ),

            "gold_answer":
                gold,

            "hallucinated_answer":
                hallu,

            "rouge_l":
                rouge_l_f1(
                    gold,
                    hallu
                ),

            "lexical_similarity":
                lexical_similarity(
                    gold,
                    hallu
                ),

            "token_change_ratio":
                token_change_ratio(
                    gold,
                    hallu
                ),

            "length_ratio":
                length_ratio(
                    gold,
                    hallu
                ),

            "change_type":
                infer_change_type(
                    gold,
                    hallu
                ),
        }

        rows.append(
            row
        )

    print(
        f"{generator}: "
        f"{len(rows):,}/{len(data):,} "
        f"valid Gold-Hallu pairs"
    )

    if skipped:

        print(
            f"  skipped = {skipped:,}"
        )

    # ========================================================
    # BERTSCORE
    # ========================================================

    if rows:

        golds = [
            x[
                "gold_answer"
            ]
            for x
            in rows
        ]

        preds = [
            x[
                "hallucinated_answer"
            ]
            for x
            in rows
        ]

        bert_scores = (
            compute_bertscore(
                golds,
                preds
            )
        )

        for (
            row,
            bs
        ) in zip(
            rows,
            bert_scores
        ):

            row[
                "bertscore_f1"
            ] = (
                float(bs)
                if not pd.isna(bs)
                else None
            )

    return rows


# ============================================================
# MODEL SUMMARY
# ============================================================

def build_model_summary(
    generator,
    generation_data,
    metric_rows,
    true_no_hallu,
    false_no_hallu
):

    total_generated = len(
        generation_data
    )

    n_true_no_hallu = len(
        true_no_hallu
    )

    n_detector_missed_hallu = len(
        false_no_hallu
    )

    # ========================================================
    # GENERATION FAILURE
    # ========================================================

    failure_rate = (
        n_true_no_hallu
        / total_generated
        if total_generated
        else 0.0
    )

    generation_success_rate = (
        1.0
        - failure_rate
    )

    # ========================================================
    # ESTIMATED TRUE HALLUCINATION
    #
    # Những sample true_no_hallu bị loại khỏi mẫu số
    # ========================================================

    estimated_hallu_count = max(
        total_generated
        - n_true_no_hallu,
        0
    )

    detector_miss_rate = (
        n_detector_missed_hallu
        / estimated_hallu_count
        if estimated_hallu_count
        else 0.0
    )

    # ========================================================
    # SIMILARITY / MINIMALITY
    # ========================================================

    if metric_rows:

        mean_rouge_l = float(
            np.mean(
                [
                    x[
                        "rouge_l"
                    ]
                    for x
                    in metric_rows
                ]
            )
        )

        mean_lexical = float(
            np.mean(
                [
                    x[
                        "lexical_similarity"
                    ]
                    for x
                    in metric_rows
                ]
            )
        )

        mean_change_ratio = float(
            np.mean(
                [
                    x[
                        "token_change_ratio"
                    ]
                    for x
                    in metric_rows
                ]
            )
        )

        mean_length_ratio = float(
            np.mean(
                [
                    x[
                        "length_ratio"
                    ]
                    for x
                    in metric_rows
                ]
            )
        )

        valid_bs = [
            x[
                "bertscore_f1"
            ]
            for x
            in metric_rows
            if x.get(
                "bertscore_f1"
            ) is not None
        ]

        mean_bertscore = (
            float(
                np.mean(
                    valid_bs
                )
            )
            if valid_bs
            else np.nan
        )

        change_types = [
            x[
                "change_type"
            ]
            for x
            in metric_rows
        ]

        diversity_entropy = (
            normalized_entropy(
                change_types
            )
        )

        change_counts = Counter(
            change_types
        )

    else:

        mean_rouge_l = 0.0
        mean_lexical = 0.0
        mean_change_ratio = 1.0
        mean_length_ratio = 0.0
        mean_bertscore = np.nan

        diversity_entropy = 0.0

        change_counts = Counter()

    return {

        "generator":
            generator,

        "total_generated":
            total_generated,

        "valid_similarity_samples":
            len(
                metric_rows
            ),

        "true_no_hallu_count":
            n_true_no_hallu,

        "generation_failure_rate":
            failure_rate,

        "generation_success_rate":
            generation_success_rate,

        "confirmed_detector_miss_count":
            n_detector_missed_hallu,

        "estimated_hallucination_count":
            estimated_hallu_count,

        "detector_miss_rate_proxy":
            detector_miss_rate,

        "mean_rouge_l":
            mean_rouge_l,

        "mean_bertscore_f1":
            mean_bertscore,

        "mean_lexical_similarity":
            mean_lexical,

        "mean_token_change_ratio":
            mean_change_ratio,

        "mean_length_ratio":
            mean_length_ratio,

        "diversity_entropy":
            diversity_entropy,

        "change_type_distribution":
            dict(
                change_counts
            ),
    }


# ============================================================
# MIN-MAX NORMALIZATION
# ============================================================

def minmax(series):

    series = series.astype(
        float
    )

    min_v = series.min()

    max_v = series.max()

    if np.isclose(
        min_v,
        max_v
    ):

        return pd.Series(
            [
                1.0
            ] * len(
                series
            ),
            index=series.index
        )

    return (
        (
            series
            - min_v
        )
        /
        (
            max_v
            - min_v
        )
    )


# ============================================================
# FINAL RANKING
# ============================================================

def compute_final_ranking(
    summary_df
):

    df = summary_df.copy()

    # ========================================================
    # GENERATION SUCCESS
    # ========================================================

    df[
        "score_generation_success"
    ] = minmax(
        df[
            "generation_success_rate"
        ]
    )

    # ========================================================
    # SEMANTIC SIMILARITY
    # ========================================================

    if (
        df[
            "mean_bertscore_f1"
        ]
        .notna()
        .all()
    ):

        semantic_raw = (
            0.70
            * df[
                "mean_bertscore_f1"
            ]
            +
            0.30
            * df[
                "mean_rouge_l"
            ]
        )

    else:

        semantic_raw = (
            0.60
            * df[
                "mean_rouge_l"
            ]
            +
            0.40
            * df[
                "mean_lexical_similarity"
            ]
        )

    df[
        "semantic_similarity_raw"
    ] = semantic_raw

    df[
        "score_semantic_similarity"
    ] = minmax(
        semantic_raw
    )

    # ========================================================
    # DETECTOR DIFFICULTY
    # ========================================================

    df[
        "score_detector_difficulty"
    ] = minmax(
        df[
            "detector_miss_rate_proxy"
        ]
    )

    # ========================================================
    # MINIMALITY
    # ========================================================

    change_good = (
        1.0
        - df[
            "mean_token_change_ratio"
        ]
    )

    minimality_raw = (
        0.5
        * change_good
        +
        0.5
        * df[
            "mean_lexical_similarity"
        ]
    )

    df[
        "minimality_raw"
    ] = minimality_raw

    df[
        "score_minimality"
    ] = minmax(
        minimality_raw
    )

    # ========================================================
    # DIVERSITY
    # ========================================================

    df[
        "score_diversity"
    ] = minmax(
        df[
            "diversity_entropy"
        ]
    )

    # ========================================================
    # FINAL SCORE
    # ========================================================

    df[
        "final_score"
    ] = (

        WEIGHTS[
            "generation_success"
        ]
        * df[
            "score_generation_success"
        ]

        +

        WEIGHTS[
            "semantic_similarity"
        ]
        * df[
            "score_semantic_similarity"
        ]

        +

        WEIGHTS[
            "detector_difficulty"
        ]
        * df[
            "score_detector_difficulty"
        ]

        +

        WEIGHTS[
            "minimality"
        ]
        * df[
            "score_minimality"
        ]

        +

        WEIGHTS[
            "diversity"
        ]
        * df[
            "score_diversity"
        ]
    )

    df = df.sort_values(
        "final_score",
        ascending=False
    ).reset_index(
        drop=True
    )

    df[
        "rank"
    ] = (
        np.arange(
            len(df)
        )
        + 1
    )

    return df


# ============================================================
# TARGET LEVEL ANALYSIS
# ============================================================

def build_target_summary(
    all_metric_rows
):

    df = pd.DataFrame(
        all_metric_rows
    )

    if df.empty:
        return pd.DataFrame()

    rows = []

    grouped = df.groupby(
        [
            "generator",
            "target_hallucination_label",
        ],
        dropna=False
    )

    for (
        generator,
        target
    ), group in grouped:

        valid_bs = (
            group[
                "bertscore_f1"
            ]
            .dropna()
        )

        rows.append(
            {

                "generator":
                    generator,

                "target_hallucination_label":
                    target,

                "n":
                    len(
                        group
                    ),

                "mean_rouge_l":
                    group[
                        "rouge_l"
                    ].mean(),

                "mean_bertscore_f1":
                    (
                        valid_bs.mean()
                        if len(
                            valid_bs
                        )
                        else np.nan
                    ),

                "mean_lexical_similarity":
                    group[
                        "lexical_similarity"
                    ].mean(),

                "mean_token_change_ratio":
                    group[
                        "token_change_ratio"
                    ].mean(),

                "mean_length_ratio":
                    group[
                        "length_ratio"
                    ].mean(),

                "diversity_entropy":
                    normalized_entropy(
                        group[
                            "change_type"
                        ].tolist()
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


# ============================================================
# DETECTOR MISS BY HALLUCINATION TYPE
# ============================================================

def build_detector_miss_by_type(
    generation_data_by_model,
    false_no_hallu_by_model,
    true_no_hallu_by_model,
):
    """
    Phân tích target hallucination type nào khiến detector bị đánh lừa.

    false_no_hallu:
        detector predicted no_hallu
        nhưng manual review xác nhận thực tế CÓ hallucination.

    true_no_hallu:
        detector predicted no_hallu
        và manual review xác nhận thực tế KHÔNG có hallucination.

    Detector miss rate(type):

        false_no_hallu(type)
        -------------------------------
        total_generated(type)
        - true_no_hallu(type)

    Lưu ý:
    đây là detector miss trên estimated true hallucination,
    vì không chạy LLM Judge toàn bộ dataset.
    """

    rows = []

    for (
        generator,
        generation_data
    ) in generation_data_by_model.items():

        false_samples = (
            false_no_hallu_by_model.get(
                generator,
                []
            )
        )

        true_samples = (
            true_no_hallu_by_model.get(
                generator,
                []
            )
        )

        # ====================================================
        # TOTAL GENERATED BY TYPE
        # ====================================================

        generated_counts = Counter()

        for item in generation_data:

            label = get_target_label(
                item
            )

            generated_counts[
                label
            ] += 1

        # ====================================================
        # DETECTOR MISS BY TYPE
        #
        # detector predicted no_hallu
        # nhưng thực tế vẫn có hallucination
        # ====================================================

        miss_counts = Counter()

        for item in false_samples:

            label = get_target_label(
                item
            )

            miss_counts[
                label
            ] += 1

        # ====================================================
        # TRUE NO HALLU BY TYPE
        #
        # generator thật sự fail sinh hallucination
        # ====================================================

        true_no_hallu_counts = Counter()

        for item in true_samples:

            label = get_target_label(
                item
            )

            true_no_hallu_counts[
                label
            ] += 1

        # ====================================================
        # BUILD RESULT
        # ====================================================

        all_labels = sorted(
            generated_counts.keys()
        )

        for label in all_labels:

            total_generated = (
                generated_counts[
                    label
                ]
            )

            n_true_no_hallu = (
                true_no_hallu_counts[
                    label
                ]
            )

            n_detector_miss = (
                miss_counts[
                    label
                ]
            )

            # Những sample ước tính thật sự có hallucination
            estimated_true_hallu = (
                total_generated
                - n_true_no_hallu
            )

            # Detector bỏ sót bao nhiêu % hallu thật
            miss_rate = (
                n_detector_miss
                / estimated_true_hallu
                if estimated_true_hallu
                else 0.0
            )

            # Generator fail theo riêng type này
            failure_rate = (
                n_true_no_hallu
                / total_generated
                if total_generated
                else 0.0
            )

            # Miss / ALL generated type
            # để tiện so với metric cũ nếu cần
            miss_pct_of_all_generated = (
                n_detector_miss
                / total_generated
                * 100
                if total_generated
                else 0.0
            )

            rows.append(
                {

                    "generator":
                        generator,

                    "hallucination_type":
                        label,

                    "total_generated":
                        total_generated,

                    "true_no_hallu":
                        n_true_no_hallu,

                    "generation_failure_rate":
                        failure_rate,

                    "generation_failure_pct":
                        failure_rate * 100,

                    "estimated_true_hallucination":
                        estimated_true_hallu,

                    "detector_miss_count":
                        n_detector_miss,

                    "detector_miss_rate":
                        miss_rate,

                    "detector_miss_pct":
                        miss_rate * 100,

                    "detector_miss_pct_of_all_generated":
                        miss_pct_of_all_generated,
                }
            )

    return pd.DataFrame(
        rows
    )


# ============================================================
# PRINT FINAL TABLE
# ============================================================

def print_final_table(
    ranking
):

    columns = [

        "rank",

        "generator",

        "total_generated",

        "true_no_hallu_count",

        "generation_failure_rate",

        "mean_rouge_l",

        "mean_bertscore_f1",

        "mean_token_change_ratio",

        "detector_miss_rate_proxy",

        "diversity_entropy",

        "final_score",
    ]

    display_df = (
        ranking[
            columns
        ]
        .copy()
    )

    # Chuyển rate -> %
    percent_cols = [

        "generation_failure_rate",

        "detector_miss_rate_proxy",
    ]

    for col in percent_cols:

        display_df[
            col
        ] = (
            display_df[
                col
            ]
            * 100
        )

    print()
    print(
        "=" * 150
    )

    print(
        "FINAL GENERATOR RANKING"
    )

    print(
        "=" * 150
    )

    print(
        display_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )


# ============================================================
# PRINT DETECTOR MISS BY TYPE
# ============================================================

def print_detector_miss_by_type(
    detector_miss_by_type
):

    if detector_miss_by_type.empty:

        print()
        print(
            "Không có dữ liệu detector miss by type."
        )

        return

    columns = [

        "generator",

        "hallucination_type",

        "total_generated",

        "true_no_hallu",

        "generation_failure_pct",

        "estimated_true_hallucination",

        "detector_miss_count",

        "detector_miss_pct",
    ]

    display_df = (
        detector_miss_by_type[
            columns
        ]
        .copy()
    )

    print()
    print()
    print(
        "=" * 150
    )

    print(
        "DETECTOR MISS BY HALLUCINATION TYPE"
    )

    print(
        "=" * 150
    )

    print(
        display_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.2f}"
        )
    )


# ============================================================
# MAIN
# ============================================================

def main():

    model_summaries = []

    all_metric_rows = []

    # ========================================================
    # DATA FOR DETECTOR MISS BY TYPE
    # ========================================================

    generation_data_by_model = {}

    true_no_hallu_by_model = {}

    false_no_hallu_by_model = {}

    # ========================================================
    # PROCESS EACH GENERATOR
    # ========================================================

    for (
        generator,
        generation_path
    ) in GENERATION_FILES.items():

        print()
        print(
            "=" * 100
        )

        print(
            f"PROCESSING {generator}"
        )

        print(
            "=" * 100
        )

        # ----------------------------------------------------
        # LOAD
        # ----------------------------------------------------

        generation_data = (
            load_json(
                generation_path
            )
        )

        true_no_hallu = (
            load_json(
                TRUE_NO_HALLU_FILES[
                    generator
                ]
            )
        )

        false_no_hallu = (
            load_json(
                FALSE_NO_HALLU_FILES[
                    generator
                ]
            )
        )

        # ----------------------------------------------------
        # SAVE FOR TYPE ANALYSIS
        # ----------------------------------------------------

        generation_data_by_model[
            generator
        ] = generation_data

        true_no_hallu_by_model[
            generator
        ] = true_no_hallu

        false_no_hallu_by_model[
            generator
        ] = false_no_hallu

        # ----------------------------------------------------
        # SIMILARITY / MINIMALITY
        # ----------------------------------------------------

        metric_rows = (
            process_generation_data(
                generator,
                generation_data
            )
        )

        all_metric_rows.extend(
            metric_rows
        )

        # ----------------------------------------------------
        # MODEL SUMMARY
        # ----------------------------------------------------

        summary = (
            build_model_summary(
                generator,
                generation_data,
                metric_rows,
                true_no_hallu,
                false_no_hallu
            )
        )

        model_summaries.append(
            summary
        )

        # ----------------------------------------------------
        # SAVE PER MODEL SAMPLE METRICS
        # ----------------------------------------------------

        pd.DataFrame(
            metric_rows
        ).to_csv(

            OUTPUT_DIR
            / f"{generator}_sample_metrics.csv",

            index=False,

            encoding="utf-8-sig"
        )

    # ========================================================
    # DETECTOR MISS BY HALLUCINATION TYPE
    # ========================================================

    detector_miss_by_type = (
        build_detector_miss_by_type(
            generation_data_by_model,
            false_no_hallu_by_model,
            true_no_hallu_by_model,
        )
    )

    detector_miss_by_type.to_csv(

        OUTPUT_DIR
        / "detector_miss_by_hallucination_type.csv",

        index=False,

        encoding="utf-8-sig"
    )

    # ========================================================
    # SAVE ALL SAMPLE METRICS
    # ========================================================

    all_metrics_df = pd.DataFrame(
        all_metric_rows
    )

    all_metrics_df.to_csv(

        OUTPUT_DIR
        / "all_sample_metrics.csv",

        index=False,

        encoding="utf-8-sig"
    )

    # ========================================================
    # MODEL SUMMARY
    # ========================================================

    summary_df = pd.DataFrame(
        model_summaries
    )

    csv_summary = (
        summary_df.copy()
    )

    csv_summary[
        "change_type_distribution"
    ] = csv_summary[
        "change_type_distribution"
    ].apply(
        lambda x: json.dumps(
            x,
            ensure_ascii=False
        )
    )

    csv_summary.to_csv(

        OUTPUT_DIR
        / "generator_metrics_summary.csv",

        index=False,

        encoding="utf-8-sig"
    )

    # ========================================================
    # FINAL RANKING
    # ========================================================

    ranking = (
        compute_final_ranking(
            summary_df
        )
    )

    ranking_csv = (
        ranking.copy()
    )

    ranking_csv[
        "change_type_distribution"
    ] = ranking_csv[
        "change_type_distribution"
    ].apply(
        lambda x: json.dumps(
            x,
            ensure_ascii=False
        )
    )

    ranking_csv.to_csv(

        OUTPUT_DIR
        / "generator_final_ranking.csv",

        index=False,

        encoding="utf-8-sig"
    )

    # ========================================================
    # TARGET ANALYSIS
    # ========================================================

    target_summary = (
        build_target_summary(
            all_metric_rows
        )
    )

    target_summary.to_csv(

        OUTPUT_DIR
        / "generator_metrics_by_target.csv",

        index=False,

        encoding="utf-8-sig"
    )

    # ========================================================
    # CHANGE TYPE DISTRIBUTION
    # ========================================================

    if not all_metrics_df.empty:

        change_distribution = (

            all_metrics_df
            .groupby(
                [
                    "generator",
                    "change_type"
                ]
            )
            .size()
            .reset_index(
                name="count"
            )
        )

        totals = (

            change_distribution
            .groupby(
                "generator"
            )[
                "count"
            ]
            .transform(
                "sum"
            )
        )

        change_distribution[
            "percentage"
        ] = (

            change_distribution[
                "count"
            ]

            / totals

            * 100
        )

        change_distribution.to_csv(

            OUTPUT_DIR
            / "change_type_distribution.csv",

            index=False,

            encoding="utf-8-sig"
        )

    # ========================================================
    # WINNER
    # ========================================================

    winner = ranking.iloc[
        0
    ]

    recommendation = {

        "selected_generator":
            winner[
                "generator"
            ],

        "rank":
            int(
                winner[
                    "rank"
                ]
            ),

        "final_score":
            float(
                winner[
                    "final_score"
                ]
            ),

        "generation_failure_rate":
            float(
                winner[
                    "generation_failure_rate"
                ]
            ),

        "mean_rouge_l":
            float(
                winner[
                    "mean_rouge_l"
                ]
            ),

        "mean_bertscore_f1":
            (
                float(
                    winner[
                        "mean_bertscore_f1"
                    ]
                )
                if not pd.isna(
                    winner[
                        "mean_bertscore_f1"
                    ]
                )
                else None
            ),

        "detector_miss_rate_proxy":
            float(
                winner[
                    "detector_miss_rate_proxy"
                ]
            ),

        "mean_token_change_ratio":
            float(
                winner[
                    "mean_token_change_ratio"
                ]
            ),

        "diversity_entropy":
            float(
                winner[
                    "diversity_entropy"
                ]
            ),

        "weights":
            WEIGHTS,

        "note":
            (
                "Ranking không sử dụng LLM Judge. "
                "Detector miss rate là proxy vì "
                "predicted no_hallu được manual review "
                "bằng is_true_no_hallu. "
                "Xem detector_miss_by_hallucination_type.csv "
                "để phân tích detector miss theo "
                "intrinsic/extrinsic/mixed."
            ),
    }

    save_json(

        recommendation,

        OUTPUT_DIR
        / "recommended_generator.json"
    )

    # ========================================================
    # PRINT FINAL RANKING
    # ========================================================

    print_final_table(
        ranking
    )

    # ========================================================
    # PRINT DETECTOR MISS BY TYPE
    # ========================================================

    print_detector_miss_by_type(
        detector_miss_by_type
    )

    # ========================================================
    # SELECTED GENERATOR
    # ========================================================

    print()
    print()

    print(
        "=" * 100
    )

    print(
        "SELECTED GENERATOR"
    )

    print(
        "=" * 100
    )

    print(
        winner[
            "generator"
        ]
    )

    print(
        f"Final score: "
        f"{winner['final_score']:.4f}"
    )

    print()

    print(
        "Output directory:"
    )

    print(
        OUTPUT_DIR
    )

    print()

    print(
        "Detector miss by hallucination type:"
    )

    print(
        OUTPUT_DIR
        / "detector_miss_by_hallucination_type.csv"
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()


PROCESSING gpt5_mini
gpt5_mini: 580/580 valid Gold-Hallu pairs

Computing BERTScore...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/72 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/74 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/37 [00:00<?, ?it/s]

done in 45.66 seconds, 12.92 sentences/sec

FINAL GENERATOR RANKING
 rank         generator  total_generated  true_no_hallu_count  generation_failure_rate  mean_rouge_l  mean_bertscore_f1  mean_token_change_ratio  detector_miss_rate_proxy  diversity_entropy  final_score
    1         gpt5_mini              580                    1                   0.1724        0.7093             0.8940                   0.2911                   12.7807             0.8961       0.8000
    2    qwen_3_8_flash              600                    6                   1.0000        0.6766             0.8788                   0.3241                   22.2222             0.8643       0.6178
    3 deepseek_v4_flash              600                   15                   2.5000        0.6316             0.8628                   0.3702                   17.4359             0.8762       0.1360


DETECTOR MISS BY HALLUCINATION TYPE
        generator      hallucination_type  total_generated  true_no_hallu  generat

# chọn sample

In [4]:
# -*- coding: utf-8 -*-

from pathlib import Path
import json
import re
import random


# ============================================================
# 1. CONFIG
# ============================================================

ROOT = Path.cwd()

DATA_DIR = ROOT / "data"

ANALYSIS_DIR = (
    ROOT
    / "generated_hallu_detector_results"
    / "analysis"
)

OUTPUT_DIR = (
    ANALYSIS_DIR
    / "human_check"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. GENERATION FILES
# ============================================================

GENERATION_FILES = {

    "gpt5_mini":
        DATA_DIR
        / "output_gpt"
        / "03_gpt5_mini_600_merged_clean.json",

    "qwen_3_8_flash":
        DATA_DIR
        / "output_qwen"
        / "03_qwen_3_8_flash_600.json",

    "deepseek_v4_flash":
        DATA_DIR
        / "output_deepseek2"
        / "03_deepseek_v4_flash_0731_600.json",
}


GENERATOR_ORDER = [
    "gpt5_mini",
    "qwen_3_8_flash",
    "deepseek_v4_flash",
]


# ============================================================
# 3. MANUALLY REVIEWED NO-HALLU FILES
# ============================================================

TRUE_NO_HALLU_FILES = {

    "gpt5_mini":
        ANALYSIS_DIR
        / "gpt5_mini_true_no_hallu.json",

    "qwen_3_8_flash":
        ANALYSIS_DIR
        / "qwen_3_8_flash_true_no_hallu.json",

    "deepseek_v4_flash":
        ANALYSIS_DIR
        / "deepseek_v4_flash_true_no_hallu.json",
}


FALSE_NO_HALLU_FILES = {

    "gpt5_mini":
        ANALYSIS_DIR
        / "gpt5_mini_false_no_hallu.json",

    "qwen_3_8_flash":
        ANALYSIS_DIR
        / "qwen_3_8_flash_false_no_hallu.json",

    "deepseek_v4_flash":
        ANALYSIS_DIR
        / "deepseek_v4_flash_false_no_hallu.json",
}


# ============================================================
# 4. HUMAN CHECK SAMPLE SIZE
# ============================================================

N_PER_TYPE = {

    "intrinsic_hallucination": 10,

    "extrinsic_hallucination": 10,

    "mixed_hallucination": 10,
}


SEED = 42

random.seed(
    SEED
)


# ============================================================
# 5. JSON UTILS
# ============================================================

def load_json(path):

    if not path.exists():

        print(
            f"[WARNING] Missing file: {path}"
        )

        return []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(
            f
        )

    if not isinstance(
        data,
        list
    ):

        raise ValueError(
            f"Expected JSON list: {path}"
        )

    return data


def save_json(
    data,
    path
):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# 6. NORMALIZATION
# ============================================================

def normalize_text(text):

    if text is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def normalize_label(label):

    label = normalize_text(
        label
    )

    mapping = {

        "intrinsic":
            "intrinsic_hallucination",

        "intrinsic_hallucination":
            "intrinsic_hallucination",

        "extrinsic":
            "extrinsic_hallucination",

        "extrinsic_hallucination":
            "extrinsic_hallucination",

        "mixed":
            "mixed_hallucination",

        "mixed_partial":
            "mixed_hallucination",

        "mixed_partial_hallucination":
            "mixed_hallucination",

        "mixed_hallucination":
            "mixed_hallucination",
    }

    return mapping.get(
        label,
        label
    )


# ============================================================
# 7. RECURSIVE FIND
# ============================================================

def recursive_find_first(
    obj,
    keys
):

    if isinstance(
        obj,
        dict
    ):

        # ưu tiên key ở current level
        for key in keys:

            value = obj.get(
                key
            )

            if (
                value is not None
                and value != ""
            ):

                return value

        # recursive xuống dưới
        for value in obj.values():

            found = recursive_find_first(
                value,
                keys
            )

            if found is not None:
                return found

    elif isinstance(
        obj,
        list
    ):

        for value in obj:

            found = recursive_find_first(
                value,
                keys
            )

            if found is not None:
                return found

    return None


# ============================================================
# 8. FIELD GETTERS
# ============================================================

def get_target_label(item):

    value = recursive_find_first(
        item,
        [
            "target_hallucination_label",
            "target_label",
        ]
    )

    return normalize_label(
        value
    )


def get_gold(item):

    value = recursive_find_first(
        item,
        [
            "gold_answer",
            "answer",
        ]
    )

    return normalize_text(
        value
    )


def get_context(item):

    value = recursive_find_first(
        item,
        [
            "context",
            "content",
        ]
    )

    return normalize_text(
        value
    )


def get_domain(item):

    return normalize_text(
        item.get(
            "domain",
            ""
        )
    )


def get_hallu(item):

    value = recursive_find_first(
        item,
        [
            "hallucinated_answer",
            "hallucinated_response",
            "generated_answer",
            "generated_response",
            "model_answer",
        ]
    )

    return normalize_text(
        value
    )


# ============================================================
# 9. GENERATION RESULT GETTERS
#
# Chỉ giữ result.
# BỎ meta hoàn toàn.
# ============================================================

def extract_result_from_generation(
    generation
):

    if not isinstance(
        generation,
        dict
    ):
        return {}

    # --------------------------------------------
    # Structure:
    #
    # {
    #   "result": {...},
    #   "meta": {...}
    # }
    #
    # -> chỉ lấy result
    # --------------------------------------------

    result = generation.get(
        "result"
    )

    if isinstance(
        result,
        dict
    ):

        return result

    # --------------------------------------------
    # Nếu bản thân object đã là result
    # --------------------------------------------

    return generation


def get_gpt_generation(item):

    generation = item.get(
        "gpt_generation",
        {}
    )

    result = extract_result_from_generation(
        generation
    )

    if not result:
        return {}

    return {
        "result":
            result
    }


def get_qwen_generation(item):

    possible_keys = [
        "qwen_generation",
        "generation",
        "generation_result",
        "result",
    ]

    for key in possible_keys:

        generation = item.get(
            key
        )

        if isinstance(
            generation,
            dict
        ):

            result = extract_result_from_generation(
                generation
            )

            if result:

                return {
                    "result":
                        result
                }

    return {}


def get_deepseek_generation(item):

    possible_keys = [
        "deepseek_generation",
        "generation",
        "generation_result",
        "result",
    ]

    for key in possible_keys:

        generation = item.get(
            key
        )

        if isinstance(
            generation,
            dict
        ):

            result = extract_result_from_generation(
                generation
            )

            if result:

                return {
                    "result":
                        result
                }

    return {}


# ============================================================
# 10. SAMPLE KEY
#
# Một chunk có thể có nhiều question.
# Vì vậy key = (chunk_id, question)
# ============================================================

def make_key(item):

    return (

        normalize_text(
            item.get(
                "chunk_id"
            )
        ),

        normalize_text(
            item.get(
                "question"
            )
        ),
    )


# ============================================================
# 11. BUILD GENERATION INDEX
# ============================================================

def build_generation_index():

    model_indexes = {}

    print()
    print(
        "=" * 100
    )

    print(
        "LOAD GENERATION FILES"
    )

    print(
        "=" * 100
    )

    for generator in GENERATOR_ORDER:

        path = GENERATION_FILES[
            generator
        ]

        data = load_json(
            path
        )

        index = {}

        duplicate_count = 0

        for item in data:

            key = make_key(
                item
            )

            if key in index:

                duplicate_count += 1

            index[
                key
            ] = item

        model_indexes[
            generator
        ] = index

        print()

        print(
            f"{generator}"
        )

        print(
            f"  rows      : {len(data):,}"
        )

        print(
            f"  unique QA : {len(index):,}"
        )

        print(
            f"  file      : {path}"
        )

        if duplicate_count > 0:

            print(
                f"  duplicates: {duplicate_count:,}"
            )

    return model_indexes


# ============================================================
# 12. PRINT FILE STRUCTURE
# ============================================================

def print_generation_structure(
    model_indexes
):

    print()
    print(
        "=" * 100
    )

    print(
        "GENERATION FILE STRUCTURE"
    )

    print(
        "=" * 100
    )

    for generator in GENERATOR_ORDER:

        index = model_indexes[
            generator
        ]

        if not index:

            print(
                f"{generator}: EMPTY"
            )

            continue

        first_item = next(
            iter(
                index.values()
            )
        )

        print()

        print(
            f"{generator}:"
        )

        print(
            list(
                first_item.keys()
            )
        )


# ============================================================
# 13. BUILD DETECTOR STATUS
# ============================================================

def build_detector_status():

    """
    detector_miss:
        Detector predicted no_hallu,
        manual review xác nhận answer thật sự CÓ hallucination.

    generation_failure:
        Detector predicted no_hallu,
        manual review xác nhận answer thật sự KHÔNG có hallucination.

    normal:
        Không nằm trong hai tập reviewed phía trên.
    """

    status_by_model = {}

    print()
    print(
        "=" * 100
    )

    print(
        "LOAD DETECTOR REVIEW STATUS"
    )

    print(
        "=" * 100
    )

    for generator in GENERATOR_ORDER:

        false_data = load_json(
            FALSE_NO_HALLU_FILES[
                generator
            ]
        )

        true_data = load_json(
            TRUE_NO_HALLU_FILES[
                generator
            ]
        )

        status = {}

        # --------------------------------------------
        # Detector miss
        # --------------------------------------------

        for item in false_data:

            key = make_key(
                item
            )

            status[
                key
            ] = "detector_miss"

        # --------------------------------------------
        # Generation failure
        # --------------------------------------------

        for item in true_data:

            key = make_key(
                item
            )

            status[
                key
            ] = "generation_failure"

        status_by_model[
            generator
        ] = status

        print()

        print(
            f"{generator}:"
        )

        print(
            f"  detector_miss      : "
            f"{len(false_data):,}"
        )

        print(
            f"  generation_failure : "
            f"{len(true_data):,}"
        )

    return status_by_model


# ============================================================
# 14. GET COMMON QA
# ============================================================

def get_common_samples(
    model_indexes
):

    common_keys = None

    for generator in GENERATOR_ORDER:

        keys = set(
            model_indexes[
                generator
            ].keys()
        )

        if common_keys is None:

            common_keys = keys

        else:

            common_keys &= keys

    if common_keys is None:

        common_keys = set()

    print()
    print(
        "=" * 100
    )

    print(
        "COMMON QA"
    )

    print(
        "=" * 100
    )

    print(
        f"Common QA across all 3 generators: "
        f"{len(common_keys):,}"
    )

    return sorted(
        common_keys
    )


# ============================================================
# 15. VERIFY TARGET LABEL ACROSS MODELS
# ============================================================

def get_common_target_label(
    key,
    model_indexes
):

    labels = {}

    for generator in GENERATOR_ORDER:

        item = model_indexes[
            generator
        ][
            key
        ]

        labels[
            generator
        ] = get_target_label(
            item
        )

    unique_labels = set(
        labels.values()
    )

    if len(
        unique_labels
    ) != 1:

        print()

        print(
            "[WARNING] Target label mismatch"
        )

        print(
            f"key = {key}"
        )

        print(
            labels
        )

        return None

    return next(
        iter(
            unique_labels
        )
    )


# ============================================================
# 16. SELECT STRATIFIED HUMAN CHECK SUBSET
# ============================================================

def select_subset(
    model_indexes,
    detector_status,
    common_keys
):

    candidates_by_type = {

        label: []

        for label
        in N_PER_TYPE
    }

    # ========================================================
    # BUILD CANDIDATE POOLS
    # ========================================================

    for key in common_keys:

        label = get_common_target_label(
            key,
            model_indexes
        )

        if label is None:

            continue

        if (
            label
            not in candidates_by_type
        ):

            continue

        statuses = {

            generator:
                detector_status[
                    generator
                ].get(
                    key,
                    "normal"
                )

            for generator
            in GENERATOR_ORDER
        }

        n_detector_miss = sum(

            status == "detector_miss"

            for status
            in statuses.values()
        )

        n_generation_failure = sum(

            status == "generation_failure"

            for status
            in statuses.values()
        )

        # ====================================================
        # PRIORITY
        #
        # Ưu tiên:
        # detector miss > generation failure > normal
        #
        # random.random() chỉ dùng để phá tie.
        # SEED cố định nên reproducible.
        # ====================================================

        priority = (

            n_detector_miss * 10

            +

            n_generation_failure * 5

            +

            random.random()
        )

        candidates_by_type[
            label
        ].append(
            {
                "key":
                    key,

                "priority":
                    priority,

                "statuses":
                    statuses,

                "n_detector_miss":
                    n_detector_miss,

                "n_generation_failure":
                    n_generation_failure,
            }
        )

    # ========================================================
    # SELECT N PER TYPE
    # ========================================================

    selected = []

    print()
    print(
        "=" * 100
    )

    print(
        "SELECT HUMAN CHECK SUBSET"
    )

    print(
        "=" * 100
    )

    for (
        label,
        n
    ) in N_PER_TYPE.items():

        candidates = candidates_by_type[
            label
        ]

        candidates = sorted(
            candidates,
            key=lambda x: x[
                "priority"
            ],
            reverse=True
        )

        chosen = candidates[
            :n
        ]

        print()

        print(
            f"{label}"
        )

        print(
            f"  candidates : {len(candidates):,}"
        )

        print(
            f"  selected   : {len(chosen):,}"
        )

        selected.extend(
            [
                {
                    "target_label":
                        label,

                    **item,
                }

                for item
                in chosen
            ]
        )

    return selected


# ============================================================
# 17. BUILD GENERATOR OUTPUT
#
# Không có human_evaluation.
# Không có meta.
# ============================================================

def build_generator_object(
    generator,
    item,
    detector_status_value
):

    # ========================================================
    # GPT-5 MINI
    # ========================================================

    if generator == "gpt5_mini":

        generation = get_gpt_generation(
            item
        )

        output = {

            "detector_review_status":
                detector_status_value,

            "gpt_generation":
                generation,
        }

        # fallback nếu generation object bị thiếu
        if not generation:

            output[
                "hallucinated_answer"
            ] = get_hallu(
                item
            )

        return output

    # ========================================================
    # QWEN 3.8 FLASH
    # ========================================================

    if generator == "qwen_3_8_flash":

        generation = get_qwen_generation(
            item
        )

        output = {

            "detector_review_status":
                detector_status_value,

            "qwen_generation":
                generation,
        }

        if not generation:

            output[
                "hallucinated_answer"
            ] = get_hallu(
                item
            )

        return output

    # ========================================================
    # DEEPSEEK V4 FLASH
    # ========================================================

    if generator == "deepseek_v4_flash":

        generation = get_deepseek_generation(
            item
        )

        output = {

            "detector_review_status":
                detector_status_value,

            "deepseek_generation":
                generation,
        }

        if not generation:

            output[
                "hallucinated_answer"
            ] = get_hallu(
                item
            )

        return output

    # ========================================================
    # FALLBACK
    # ========================================================

    return {

        "detector_review_status":
            detector_status_value,

        "hallucinated_answer":
            get_hallu(
                item
            ),
    }


# ============================================================
# 18. BUILD FINAL HUMAN CHECK JSON
# ============================================================

def build_human_check_json(
    selected,
    model_indexes,
    detector_status
):

    results = []

    for (
        sample_id,
        selected_item
    ) in enumerate(
        selected,
        start=1
    ):

        key = selected_item[
            "key"
        ]

        chunk_id, question = key

        # ====================================================
        # BASE QA
        #
        # Dùng GPT CLEAN làm base.
        # ====================================================

        base_item = model_indexes[
            "gpt5_mini"
        ][
            key
        ]

        target_label = get_common_target_label(
            key,
            model_indexes
        )

        sample = {

            "sample_id":
                sample_id,

            "chunk_id":
                chunk_id,

            "doc_id":
                normalize_text(
                    base_item.get(
                        "doc_id",
                        ""
                    )
                ),

            "domain":
                get_domain(
                    base_item
                ),

            "context":
                get_context(
                    base_item
                ),

            "question":
                question,

            "gold_answer":
                get_gold(
                    base_item
                ),

            "target_hallucination_label":
                target_label,

            "generations":
                {}
        }

        # ====================================================
        # ADD ALL 3 GENERATORS
        # ====================================================

        for generator in GENERATOR_ORDER:

            item = model_indexes[
                generator
            ][
                key
            ]

            status_value = detector_status[
                generator
            ].get(
                key,
                "normal"
            )

            sample[
                "generations"
            ][
                generator
            ] = build_generator_object(

                generator=
                    generator,

                item=
                    item,

                detector_status_value=
                    status_value
            )

        results.append(
            sample
        )

    return results


# ============================================================
# 19. PRINT SUMMARY
# ============================================================

def print_subset_summary(
    review_json
):

    print()
    print(
        "=" * 100
    )

    print(
        "HUMAN CHECK SUBSET SUMMARY"
    )

    print(
        "=" * 100
    )

    print(
        f"Unique QA: "
        f"{len(review_json):,}"
    )

    print(
        f"Total model outputs: "
        f"{len(review_json) * len(GENERATOR_ORDER):,}"
    )

    # ========================================================
    # TARGET DISTRIBUTION
    # ========================================================

    target_counts = {}

    for sample in review_json:

        label = sample[
            "target_hallucination_label"
        ]

        target_counts[
            label
        ] = (
            target_counts.get(
                label,
                0
            )
            + 1
        )

    print()

    print(
        "Target distribution:"
    )

    for (
        label,
        count
    ) in target_counts.items():

        print(
            f"  {label}: {count}"
        )

    # ========================================================
    # DETECTOR STATUS DISTRIBUTION
    # ========================================================

    print()

    print(
        "Detector status in selected subset:"
    )

    for generator in GENERATOR_ORDER:

        counts = {

            "normal":
                0,

            "detector_miss":
                0,

            "generation_failure":
                0,
        }

        for sample in review_json:

            status = (
                sample[
                    "generations"
                ][
                    generator
                ][
                    "detector_review_status"
                ]
            )

            counts[
                status
            ] = (
                counts.get(
                    status,
                    0
                )
                + 1
            )

        print()

        print(
            f"  {generator}:"
        )

        print(
            f"    normal             = "
            f"{counts.get('normal', 0)}"
        )

        print(
            f"    detector_miss      = "
            f"{counts.get('detector_miss', 0)}"
        )

        print(
            f"    generation_failure = "
            f"{counts.get('generation_failure', 0)}"
        )


# ============================================================
# 20. VALIDATE OUTPUT
# ============================================================

def validate_output(
    review_json
):

    print()
    print(
        "=" * 100
    )

    print(
        "VALIDATE OUTPUT"
    )

    print(
        "=" * 100
    )

    problems = []

    for sample in review_json:

        sample_id = sample[
            "sample_id"
        ]

        generations = sample[
            "generations"
        ]

        # ----------------------------------------------------
        # đủ 3 generator?
        # ----------------------------------------------------

        for generator in GENERATOR_ORDER:

            if generator not in generations:

                problems.append(
                    f"sample {sample_id}: "
                    f"missing {generator}"
                )

        # ----------------------------------------------------
        # kiểm tra result
        # ----------------------------------------------------

        key_map = {

            "gpt5_mini":
                "gpt_generation",

            "qwen_3_8_flash":
                "qwen_generation",

            "deepseek_v4_flash":
                "deepseek_generation",
        }

        for (
            generator,
            generation_key
        ) in key_map.items():

            generation = generations.get(
                generator,
                {}
            ).get(
                generation_key,
                {}
            )

            if not generation:

                problems.append(
                    f"sample {sample_id}: "
                    f"missing {generation_key}"
                )

                continue

            # ------------------------------------------------
            # meta không được tồn tại
            # ------------------------------------------------

            if (
                isinstance(
                    generation,
                    dict
                )
                and "meta" in generation
            ):

                problems.append(
                    f"sample {sample_id}: "
                    f"{generation_key} still contains meta"
                )

    if problems:

        print(
            f"Found {len(problems)} problem(s):"
        )

        for problem in problems[
            :20
        ]:

            print(
                f"  - {problem}"
            )

        if len(problems) > 20:

            print(
                f"  ... and "
                f"{len(problems) - 20} more"
            )

    else:

        print(
            "OK"
        )

        print(
            "- All selected QA have 3 generators."
        )

        print(
            "- Generation result objects found."
        )

        print(
            "- No meta objects remain."
        )


# ============================================================
# 21. MAIN
# ============================================================

def main():

    # ========================================================
    # LOAD GENERATIONS
    # ========================================================

    model_indexes = (
        build_generation_index()
    )

    # ========================================================
    # SHOW ORIGINAL FILE STRUCTURE
    # ========================================================

    print_generation_structure(
        model_indexes
    )

    # ========================================================
    # LOAD DETECTOR REVIEW
    # ========================================================

    detector_status = (
        build_detector_status()
    )

    # ========================================================
    # COMMON QA
    # ========================================================

    common_keys = (
        get_common_samples(
            model_indexes
        )
    )

    if not common_keys:

        raise RuntimeError(
            "Không tìm thấy QA chung giữa 3 generator."
        )

    # ========================================================
    # SELECT SUBSET
    # ========================================================

    selected = (
        select_subset(
            model_indexes,
            detector_status,
            common_keys
        )
    )

    # ========================================================
    # BUILD FINAL JSON
    # ========================================================

    review_json = (
        build_human_check_json(
            selected,
            model_indexes,
            detector_status
        )
    )

    # ========================================================
    # VALIDATE
    # ========================================================

    validate_output(
        review_json
    )

    # ========================================================
    # SAVE
    # ========================================================

    output_path = (
        OUTPUT_DIR
        / "human_check_subset.json"
    )

    save_json(
        review_json,
        output_path
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    print_subset_summary(
        review_json
    )

    print()
    print(
        "=" * 100
    )

    print(
        "SAVED"
    )

    print(
        "=" * 100
    )

    print(
        output_path.resolve()
    )


# ============================================================
# 22. RUN
# ============================================================

if __name__ == "__main__":

    main()


LOAD GENERATION FILES

gpt5_mini
  rows      : 580
  unique QA : 580
  file      : /home/trang/study/khoaluan/KLTN/src/create_hallu/qa/data/output_gpt/03_gpt5_mini_600_merged_clean.json

qwen_3_8_flash
  rows      : 600
  unique QA : 600
  file      : /home/trang/study/khoaluan/KLTN/src/create_hallu/qa/data/output_qwen/03_qwen_3_8_flash_600.json

deepseek_v4_flash
  rows      : 600
  unique QA : 600
  file      : /home/trang/study/khoaluan/KLTN/src/create_hallu/qa/data/output_deepseek2/03_deepseek_v4_flash_0731_600.json

GENERATION FILE STRUCTURE

gpt5_mini:
['chunk_id', 'doc_id', 'domain', 'content', 'question', 'answer', 'target_hallucination_label', 'gpt_generation']

qwen_3_8_flash:
['chunk_id', 'doc_id', 'domain', 'content', 'question', 'answer', 'target_hallucination_label', 'qwen_generation']

deepseek_v4_flash:
['chunk_id', 'doc_id', 'domain', 'content', 'question', 'answer', 'target_hallucination_label', 'deepseek_generation']

LOAD DETECTOR REVIEW STATUS

gpt5_mini:
  detect